<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EA%B3%A0%EA%B8%89_RAG_%EB%A6%AC%EB%9E%AD%ED%82%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경설정

In [1]:
!pip install langchain langchain_openai langchain_community pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
os.environ['OPENAI_API_KEY'] = None

In [3]:
!wget "https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf"

--2026-02-27 10:38:31--  https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf [following]
--2026-02-27 10:38:31--  https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2242511 (2.1M) [application/octet-stream]
Sav

#LLM 기반 리랭킹
텍스트 불러오기, Chunking, VectorDB 저장

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('2023_북한인권보고서.pdf')
doc_splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 100)

docs = loader.load_and_split(doc_splitter)

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')
faiss_store = FAISS.from_documents(docs, embedding)

save_dir = './DB'
faiss_store.save_local(save_dir)
#saved_faiss_store = FAISS.load_local(save_dir, embeddings = embedding, allow_dangerous_deserialization = True)



#리랭킹 알고리즘

LLM 사용한 리랭킹 함수 만들기

In [70]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

from typing import List
from langchain_core.documents import Document

class RelevanceScore(BaseModel):
    relevance_score: float = Field(description="문서가 쿼리와 얼마나 관련이 있는지를 나타내는 점수.")

def reranking_documents(query : str, docs : List[Document], top_n : int = 2) -> List[Document]:

  prompt = ChatPromptTemplate.from_template("""
  1점부터 10점까지 점수를 매겨서 다음 문서가 질문과 얼마나 관련이 있는지를 평가해주세요.
  단순히 키워드가 일치하는 것이 아니라 쿼리의 구체적인 맥락과 의도를 고려하세요.
  {format_instructions}

  question: {query}
  document: {doc}
  relevance_score:
  """)

  llm = ChatOpenAI(model = 'gpt-4o', temperature = 0, max_tokens = 3000)

  parser = JsonOutputParser(pydantic_object = RelevanceScore)

  result = []

  for doc in docs:
    try:
      formatted = prompt.format_messages(
          query = query,
          doc = doc.page_content,
          format_instructions = parser.get_format_instructions()
      )

      llm_response = llm.invoke(formatted)
      parsed = parser.invoke(llm_response)

      score = parsed.get('relevance_score', 5)


    except Exception as e:
      print(f'오류 발생 : {e}, 기본 점수 5점 부여')
      score = 5.0

    result.append((doc, score))

  result = sorted(result, key = lambda x : x[1], reverse = True)
  result = [doc for doc, _ in result[:top_n]]

  return result

첫 검색, 리랭킹 비교하기

In [71]:
query = '북한 인권실태에 관한 형량 재판 실례'

initial_docs = faiss_store.similarity_search(query, k=4)
reranked_docs = reranking_documents(query, initial_docs)

In [72]:
# print first 4 initial documents
print(f"Query: {query}\n\n")

print("Top initial documents:")
for i, doc in enumerate(initial_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

# Print results
print("\n\nTop reranked documents:")
for i, doc in enumerate(reranked_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)  # Print each documents

Query: 북한 인권실태에 관한 형량 재판 실례


Top initial documents:

Document 1:
등 재판관계자만 참석할 수 있도록 재판이 비공개로 진행되어 가족
들이 참관할 수 없었다는 증언이 있었다. 진술에 따르면 판결의 선
고도 공개되지 않았다고 한다. 
(2) 현지공개재판
재판의 심리와 판결의 공개는 공정한 재판을 받을 권리의 기본적
인 구성 요소이다. 그런데 북한이탈주민들의 증언에 따르면 북한에
서는 공개재판제도를 주민교양을 위한 선전도구로 이용하는 있는 것
으로 나타났다. 그러한 제도 중의 하나로 ‘현지공개재판’142이 있는데, 
이는 군중을 각성시키고 범죄를 예방하기 위하여 재판소가 현지에서

Document 2:
7. 공정한 재판을 받을 권리
149
IV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리
모든 국가기관에도 인정되며, 국가기관은 피고인의 유죄를 단언하
는 공식 발표를 삼가는 등 재판 결과를 예단해서는 안 된다.145 
북한에서는 피의자의 무죄로 추정받을 권리가 제대로 보장되지 
않는 것으로 나타났다. 한 증언자는 2019년에 안전부 구류장에 비
법국제통신죄로 구금되어 있을 때, 최고인민회의 대의원 선거가 있
었는데, 유죄판결을 받지않았음에도 불구하고 안전부에서 교화형

Document 3:
재판심리를 공개하지 않는 경우에도 판결의 선고는 공개한다고 주
장하였다.141 그러나 북한에서 재판을 받은 적이 있는 북한이탈주민
들의 증언에 따르면 북한의 공개재판 원칙은 제대로 지켜지지 않는 
137	 	「사회주의헌법(2019)」	제164조.
138	 	북한	백과사전출판사	『광명백과사전』,	2009,	612쪽.
139	 	「사회주의헌법(2019)」	제164조.
140	  「형사소송법(2021)」 제267조.
141	 	UN	Doc.	A/HRC/WG.6/33/PRK/1	(2019),	para.	28.

Document 4:
2023 북한인권보고서
146
심리를 받을 

커스텀 리트리버 클래스 생성하기

In [87]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import Any

class CustomRetriever(BaseRetriever):

  vectorstore: Any = None # BaseRetriever는 pydantic 모델 기반이라 타입 필드 선언이 필요함

  def __init__(self, vectorstore : Any):
    super().__init__()
    self.vectorstore = vectorstore

  def _get_relevant_documents(self, query : str) -> List[Document]:

    initial_docs = self.vectorstore.similarity_search(query, k = 4)
    reranked_docs = reranking_documents(query, initial_docs, top_n = 2)

    return reranked_docs

  async def _aget_relevant_documents(self, query: str) -> List[Document]:
    raise NotImplementedError("Async retrieval not implemented")

RAG 함수 사용

In [112]:
custom_retriever = CustomRetriever(faiss_store)

llm = ChatOpenAI(model = 'gpt-4.1', temperature=0.2)

prompt_template = ChatPromptTemplate.from_template("""
주어진 검색 결과를 바탕으로 사용자의 질문에 정확하고 자연스럽게 답변하세요.
검색 결과에 질문에 관한 내용이 없으면 솔직히 모른다고 답변하세요.

검색 결과 :
{context}

사용자 질문 :
{query}

답변 :
""")

def query_RAG(query : str):
  docs = custom_retriever.invoke(query)
  context = "\n\n".join([doc.page_content for doc in docs])

  prompt = prompt_template.format_messages(query = query, context = context)

  response = llm.invoke(prompt)

  return {
      'query' : query,
      'result' : response.content,
      'source_documents' : context
  }

In [115]:
answer = query_RAG('북한 여자')

In [116]:
for k in answer:
  print(k)
  print(answer[k])

  print()

query
북한 여자

result
검색 결과에 따르면, 북한 여성들은 경제적인 이유로 중국으로 탈북하는 경우가 많습니다. 북한 내에서 국가 배급이 제대로 이루어지지 않아 생계를 위해 중국으로 넘어가려는 사람들이 많으며, 특히 여성과 아동들이 인신매매의 대상이 되는 사례가 보고되고 있습니다. 중국에 도착한 후에는 조선족 인신매매 브로커에 의해 팔려가는 경우도 있습니다.

또한, 중국에서 체포되어 북한으로 강제 송환된 여성들은 노동교화형과 같은 처벌을 받기도 하며, 구류장에서는 계호원들에 의한 성희롱이나 벌을 받는 등의 인권 침해 사례도 다수 보고되고 있습니다. 

요약하자면, 북한 여성들은 경제적 어려움으로 인해 탈북을 시도하지만, 그 과정에서 인신매매, 강제 송환, 인권 침해 등 다양한 위험에 노출되어 있습니다.

source_documents
1. 여성
377
IV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리
쳐 이동하던 중 체포된 여성은 인신매매 피해자였기 때문에 북한으
로 송환되면 처벌을 받지 않을 것이라고 생각하였지만, 2015년 말 
재판을 받은 결과 노동교화형이 결정되었다고 진술하였다. 그리고 
구류장에서 계호원 등이 강제송환된 여성들만 벌을 주거나 성희롱
을 일삼았다는 다수의 증언이 수집되었다. 
“탈북한 이유는 돈을 벌기 위한 경제적인 목적이었고, 팔려간다는

1. 여성
375
IV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리
“살기 위해 중국으로 가고자 했습니다. 국가 배급도 전혀 없었고  
중국에 가면 먹고 살 수 있다는 소문만 믿고 중국으로 탈북하게 되
었습니다. 함경북도 청진역전에 ‘중국에 가면 돈을 벌수도 있고 배
불리 살 수도 있다’면서 접근하는 40대 여성들의 말을 믿고 중국으
로 팔려가는 북한 어린 여자들과 아동들이 많았습니다. 저도 그중
에 한 명입니다. 중국에 도착하니 조선족 인신매매브로커가 있었고,

#크로스 인코더 기반 리랭킹

In [117]:
!wget 'https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf'

--2026-02-27 12:46:54--  https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf [following]
--2026-02-27 12:46:55--  https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2242511 (2.1M) [application/octet-stream]
Sav

In [118]:
loader = PyPDFLoader('2023_북한인권보고서.pdf')
doc_splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 100)

docs = loader.load_and_split(doc_splitter)

In [121]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

faiss_store = FAISS.from_documents(docs, embedding)
faiss_store.save_local('./DB2')

In [124]:
vector_db = FAISS.load_local('./DB2', embeddings = embedding, allow_dangerous_deserialization=True)

In [125]:
from sentence_transformers import CrossEncoder
crossencoder = CrossEncoder('BAAI/bge-reranker-v2-m3')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

cross encoder 기반 커스텀 리트리버

In [150]:
class CrossEncoderRetriever(BaseRetriever):
  vectorstore : Any
  crossencoder : Any
  k : int = 5
  rerank_top_k : int = 2

  def _get_relevant_documents(self, query : str) -> List[Document]:
    initial_docs = self.vectorstore.similarity_search(query, k=self.k)

    pairs = [[query, doc.page_content] for doc in initial_docs]

    scores = crossencoder.predict(pairs)
    scored_docs = sorted(zip(initial_docs, scores), key = lambda x : x[1], reverse = True)

    return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

  async def _aget_relevant_documents(self, query: str) -> List[Document]:
      raise NotImplementedError("Async retrieval not implemented")

In [153]:
cross_encoder_retriever = CrossEncoderRetriever(
    vectorstore = vector_db,
    crossencoder = crossencoder,
    k = 5,
    rerank_top_k = 2
)

llm = ChatOpenAI(model = 'gpt-4o', temperature = 0.2)

In [159]:
def query_rag(query):
  docs = cross_encoder_retriever.invoke(query)
  context = "\n\n".join([doc.page_content for doc in docs])

  prompt = ChatPromptTemplate.from_template("""
당신은 북한 경제 관련 문서를 기반으로 정보를 제공하는 분석 도우미입니다.
다음 검색 결과를 참고하여 질문에 답변하세요.
정보가 부족하면 '모르겠습니다'라고 답변하세요.

검색 결과:
{context}

질문: {query}

답변:
""")
  messages = prompt.format_messages(query=query, context=context)

  # 4. LLM 응답 생성
  response = llm.invoke(messages)

  # 5. 결과 반환
  return {
      "query": query,
      "result": response.content,
      "source_documents": docs
  }


In [160]:
answer = query_rag('여성인권')
print(answer)

{'query': '여성인권', 'result': '여성 인권은 성별과 관계없이 모든 사람에게 동등하게 보장되어야 합니다. 세계인권선언에서는 모든 인류 구성원에 대한 존엄성과 평등을 강조하며, 성별에 따른 차별 없이 모든 권리와 자유를 누릴 자격이 있다고 명시하고 있습니다. 북한은 여성차별철폐협약을 비준하였으며, 협약 이행에 대한 보고서를 제출하는 등 여성 인권 증진과 보호를 위한 노력을 하고 있습니다. 그러나 구체적인 상황이나 효과에 대한 정보는 제공되지 않았습니다.', 'source_documents': [Document(id='aac858f2-dfe0-4bd3-ac5c-0b2d3363eb1f', metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2023-07-31T13:50:27+09:00', 'moddate': '2023-07-31T13:57:54+09:00', 'trapped': '/False', 'source': '2023_북한인권보고서.pdf', 'total_pages': 448, 'page': 365, 'page_label': '366'}, page_content='2023 북한인권보고서\n364\n1. 여성\n인권은 성별과 관계없이 누구에게나 동등하게 보장되어야 한다. \n세계인권선언의 전문에서는 모든 인류 구성원에 대한 고유한 존엄\n성과 평등을 강조하며, 제2조에서는 “모든 사람은 성별을 비롯하여 \n어떠한 이유에 의해서도 차별받지 않고 선언에 제시된 모든 권리와 \n자유를 누릴 자격이 있다.”라고 선언하고 있다. 자유권규약과 사회\n권규약에서도 남녀 모두에게 동등한 권리가 있음을 확인하고, 규약\n의 당사국은 규약에서 명시하고 있는 모든 권리를 남녀에게 동등하'), Document(id='dd6a7930-bf48-4bf4-816e-81cbb3455f96', metadata={'producer':

In [163]:
print(f'query : {answer['query']}')
print(f'result : {answer['result']}')

query : 여성인권
result : 여성 인권은 성별과 관계없이 모든 사람에게 동등하게 보장되어야 합니다. 세계인권선언에서는 모든 인류 구성원에 대한 존엄성과 평등을 강조하며, 성별에 따른 차별 없이 모든 권리와 자유를 누릴 자격이 있다고 명시하고 있습니다. 북한은 여성차별철폐협약을 비준하였으며, 협약 이행에 대한 보고서를 제출하는 등 여성 인권 증진과 보호를 위한 노력을 하고 있습니다. 그러나 구체적인 상황이나 효과에 대한 정보는 제공되지 않았습니다.
